# Cod3x Trainer
Train on 50K factory-generated examples. Built by Codex Developer.

In [ ]:
# 1. Clone
import os
if not os.path.exists('Cod3x'):
    !git clone https://github.com/codexhaven/Cod3x.git
%cd Cod3x


In [ ]:
# 2. Install
!pip install -q transformers datasets peft accelerate torch huggingface_hub


In [ ]:
# 3. HF Token
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded')
except:
    print('Set HF_TOKEN in Colab secrets (key icon)')


In [ ]:
# 4. Load 50K training data
import json
with open('data/training_data.json') as f:
    data = json.load(f)
print(f'Loaded {len(data)} training examples')
print(f'Sample: {data[0]["instruction"][:80]}')


In [ ]:
# 5. Train
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from transformers import Trainer, DataCollatorForLanguageModeling

BASE='Qwen/Qwen2.5-0.5B-Instruct'
OUT='./persona_output'
PNAME='cod3x-default'

print(f'Loading {BASE}...')
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32, device_map='cpu')
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token

with open('data/training_data.json') as f:
    data = json.load(f)

def fmt(ex):
    m=[{'role':'system','content':f'You are {PNAME}, trained by Cod3x.'},{'role':'user','content':ex['instruction']},{'role':'assistant','content':ex['response']}]
    return {'text':tok.apply_chat_template(m, tokenize=False)}

ds = Dataset.from_list(data).map(fmt)
ds = ds.map(lambda ex: tok(ex['text'], truncation=True, max_length=512), batched=True, remove_columns=['text'])

lc = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj','v_proj'], lora_dropout=0.1, bias='none', task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lc)
model.print_trainable_parameters()

ta = TrainingArguments(output_dir=OUT, per_device_train_batch_size=1, gradient_accumulation_steps=4, num_train_epochs=3, learning_rate=2e-4, logging_steps=50, save_strategy='epoch', report_to='none')
tr = Trainer(model=model, args=ta, train_dataset=ds, data_collator=DataCollatorForLanguageModeling(tok, mlm=False))

print(f'Training on CPU with {len(data)} examples...')
tr.train()
model.save_pretrained(OUT)
tok.save_pretrained(OUT)
print(f'Saved to {OUT}')


In [ ]:
# 6. Upload
from huggingface_hub import login, upload_folder
login()
REPO='codexhaven/cod3x-persona'
upload_folder(folder_path='./persona_output', repo_id=REPO, repo_type='model')
print(f'Live: https://huggingface.co/{REPO}')
